In [1]:
#Plant and controller
import numpy as np
import gapMetrics
import bacteriaSys

In [9]:
# make a function that gets the Hankel matrix
def get_Hankel_from_io(param):
    """
    param: the value of the constant parameter in the bacteria system
    """
    # simulate the bacteria system
    x_star = np.zeros(18)  # nominal state
    u_star = np.array([1.0, 1.0])  # nominal input
    Ts = 1  # sampling time
    Ad, Bd, Cd = bacteriaSys.linearised_discrete_system(x_star, u_star, Ts, p=param)
    D = np.zeros((Cd.shape[0], Bd.shape[1]))  # direct transmission matrix
    T = 1000  # length of the simulation
    u = np.random.rand(T,Bd.shape[1])  # random input
    _, y = gapMetrics.simulate_system(Ad, Bd, Cd, D, u, x0=None)

    # build the Hankel matrix from input-output data
    L = 20  # block horizon
    H_y = gapMetrics.hankel_matrix(y, L)
    H_u = gapMetrics.hankel_matrix(u, L)
    H = np.vstack([H_u, H_y])

    return H

In [17]:
param1 = bacteriaSys.set_params()
H1 = get_Hankel_from_io(param=param1)
param2 = bacteriaSys.set_params(mu_g=0.02,gamma_max=30.0)
H2 = get_Hankel_from_io(param=param2)

# Compute L-gap metric
Lgap, _ = gapMetrics.Lgap_metric(H1, H2)
print("L-gap metric between the two systems:", Lgap)

L-gap metric between the two systems: 4.214684851089402e-08
